In [1]:
!pip install cugraph-cu12==25.2.0 -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import cudf
import cugraph
import cupy as cp

In [2]:
import pandas as pd
import numpy as np
import requests
import io

In [7]:
# Step 1: URL of the raw .gdf file
url = "https://raw.githubusercontent.com/amirkasaei/Social-and-Economic-Networks/refs/heads/main/HW/HW2/HW2-dataset.gdf"

# Download the file content
response = requests.get(url)
gdf_data = response.text.splitlines()

# Parse edges
edge_lines = []
reading_edges = False
for line in gdf_data:
    line = line.strip()
    if line.startswith('edgedef>'):
        reading_edges = True
        edge_header = line.replace('edgedef>', '').strip()
        continue
    if reading_edges:
        edge_lines.append(line)

# Convert edges into pandas DataFrame
edge_csv = "\n".join(edge_lines)

edge_df = pd.read_csv(io.StringIO(edge_csv), header=None)

# Extract correct columns
columns = [col.split()[0] for col in edge_header.split(',')]
edge_df.columns = columns

print("✅ Parsed edges from GitHub:")
print(edge_df.head())

✅ Parsed edges from GitHub:
                 node1                node2  directed  weight
0  1000064596290424832  1411377479634276358      True       2
1  1000064596290424832  1445444813143216140      True       1
2  1000064596290424832  1743513189600391168      True       1
3  1000360620112433152  1147080345067687936      True       1
4  1000360620112433152  1234568599785943050      True       1


In [8]:
### MOVE DATA TO GPU (CUDF)

# Convert pandas DataFrame to cuDF DataFrame
edge_cudf = cudf.DataFrame.from_pandas(edge_df)

# edge_cudf['node1'] = edge_cudf['node1'].astype('int32')
# edge_cudf['node2'] = edge_cudf['node2'].astype('int32')
# edge_cudf['weight'] = edge_cudf['weight'].astype('float32')


In [9]:
### BUILD GRAPH ON GPU

# Create a directed graph
G = cugraph.Graph(directed=True)  # ← instead of DiGraph()

# Force node1 and node2 to int32
edge_cudf['node1'] = edge_cudf['node1'].astype('int32')
edge_cudf['node2'] = edge_cudf['node2'].astype('int32')


# Then build graph normally:
G.from_cudf_edgelist(edge_cudf, source='node1', destination='node2', edge_attr='weight')
print(f"\nGraph has {G.number_of_vertices()} vertices and {G.number_of_edges()} edges.")


Graph has 12588 vertices and 53580 edges.


In [10]:
### PART 4: SPECTRAL CLUSTERING ON GPU

# Perform spectral clustering
num_clusters = 3  # Adjust as needed
clusters = cugraph.spectralModularityMaximizationClustering(G, num_clusters)

print(clusters.head())

       vertex  cluster
0   257327105        2
1  1792438273        2
2 -1519650169        0
3  -999861379        2
4   219652096        0


In [12]:
### PART 5: CENTRALITY (Important nodes)

# Compute PageRank as a centrality measure
pagerank_df = cugraph.pagerank(G)

# Merge clusters and PageRank scores
result = clusters.merge(pagerank_df, on='vertex')

# Find important node per cluster (highest pagerank in each cluster)
important_nodes_per_cluster = {}
k=3

for cluster_id in range(k):
    cluster_nodes = result[result['cluster'] == cluster_id]
    important_node = cluster_nodes.sort_values('pagerank', ascending=False).iloc[0]
    important_nodes_per_cluster[cluster_id] = int(important_node['vertex'])

print("\nImportant nodes per cluster (highest PageRank):")
for cluster_id, node in important_nodes_per_cluster.items():
    print(f"Cluster {cluster_id}: Important Node -> {node}")

/usr/local/lib/python3.11/dist-packages/cugraph/link_analysis/pagerank.py:232: UserWarning: Pagerank expects the 'store_transposed' flag to be set to 'True' for optimal performance during the graph creation
  warnings.warn(warning_msg, UserWarning)



Important nodes per cluster (highest PageRank):
Cluster 0: Important Node -> -1537761280
Cluster 1: Important Node -> 836227072
Cluster 2: Important Node -> 584359936
